# CS 521 HW3, Problem 3

[Open in Colab](https://colab.research.google.com/drive/1LZaWXeM-3KrDwS-h1439f0bd2xjf143I)

Locally, select `hw3/.venv` and confirm `device: mps`. In Colab, use T4 and runtime 2026.07.
Each training cell starts a fresh model from the supplied checkpoint.

Saved outputs are from the completed Apple M5 Pro run before the driver was simplified.
The objectives and training settings are unchanged. Original outputs and checkpoints are preserved in `results/mps/`.


In [ ]:
from pathlib import Path
import gdown


def find_hw3_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'q3').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
        if (candidate / 'hw3' / 'q3').is_dir():
            return candidate / 'hw3'
    # A standalone Colab upload uses the runtime's current directory.
    return cwd


HW3_ROOT = find_hw3_root()
DATA_DIR = HW3_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FILES = [
    ('17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes', 'linf_base.pt'),
    ('1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe', 'cifar10_train_10k.pt'),
    ('1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA', 'cifar10_test_1k.pt'),
]
for file_id, filename in FILES:
    destination = DATA_DIR / filename
    if not destination.is_file():
        partial = destination.with_suffix('.pt.part')
        downloaded = gdown.download(id=file_id, output=str(partial), quiet=False, use_cookies=False)
        if downloaded is None:
            raise RuntimeError(f'Download failed: {filename}. Place the assignment file in {DATA_DIR}')
        partial.replace(destination)
print('Shared data:', DATA_DIR)

Shared data: /Users/zhiyuwu2/uiuc/CCS521FA26HW/hw3/data


In [ ]:
import copy
import os
import platform
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

automatic_device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device = torch.device(os.environ.get("HW3_DEVICE", automatic_device))
if device.type == "mps" and not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable in this process. Run from a local Terminal or Jupyter kernel with GPU access.")
print("device:", device, "torch:", torch.__version__, "platform:", platform.platform())

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: mps torch: 2.14.1 platform: macOS-26.7.1-arm64-arm-64bit


In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(DATA_DIR / path, map_location="cpu")
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load(DATA_DIR / "linf_base.pt", map_location=device))
    return model

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta).detach()
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        radius = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta).clamp_min(1e-12) * eps * radius
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp_max(1)
        delta = clip_box(x, delta).detach()
    return (x + delta).detach()

In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    model.epoch_times, model.train_losses = [], []
    for epoch in range(epochs):
        if device.type == "mps":
            torch.mps.synchronize()
        elif device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        if device.type == "mps":
            torch.mps.synchronize()
        elif device.type == "cuda":
            torch.cuda.synchronize()
        seconds = time.perf_counter() - start
        model.epoch_times.append(seconds)
        model.train_losses.append(float(np.mean(losses)))
        print(f"epoch {epoch + 1}/{epochs}: loss {np.mean(losses):.4f}, {seconds:.1f} s")
    model.eval()
    return model

In [ ]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q, logits_r = model(x_q), model(x_r)
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    correct_q = (logits_q.argmax(1) == y).to(logits_q.dtype)
    # KL(p_q || p_r), with gradients through both predictions.
    kl = F.kl_div(F.log_softmax(logits_r, dim=1),
                  F.softmax(logits_q, dim=1), reduction="none").sum(1)
    pairing = (kl * correct_q).sum() / correct_q.sum().clamp_min(1)
    return torch.maximum(ce_q, ce_r).mean() + lam * pairing


def gradient_projection(g_n, g_a):
    # The assignment specifies positive cosine times g_n, tensor by tensor.
    denom = g_n.norm() * g_a.norm()
    cosine = (g_n * g_a).sum() / denom.clamp_min(1e-12)
    return cosine.clamp(0, 1) * g_n

In [ ]:
class AdaptiveNormDistillation:
    def __init__(self, teacher, distill_weight=0.5, ema_decay=0.9):
        self.teacher = teacher.eval().requires_grad_(False)
        self.distill_weight = distill_weight
        self.ema_decay = ema_decay
        self.step = 0
        self.errors = {norm: None for norm in NORMS}
        self.trace = []

    def __call__(self, model, x, y):
        slot = self.step % 4
        norm = NORMS[slot] if slot < 3 else max(NORMS, key=lambda p: self.errors[p])
        model.eval()
        x_adv = attack(model, x, y, norm)
        with torch.no_grad():
            failure = (model(x_adv).argmax(1) != y).float().mean().item()
            previous = self.errors[norm]
            self.errors[norm] = failure if previous is None else (
                self.ema_decay * previous + (1 - self.ema_decay) * failure)
            if self.distill_weight:
                self.teacher.eval()
                teacher_logits = self.teacher(x_adv)
                target = F.softmax(teacher_logits, dim=1)
                correct = (teacher_logits.argmax(1) == y).to(x.dtype)
        model.train()
        logits = model(x_adv)
        ce = F.cross_entropy(logits, y)
        anchor = logits.sum() * 0
        gate_fraction = 0.0
        if self.distill_weight:
            kl = F.kl_div(F.log_softmax(logits, dim=1), target, reduction="none").sum(1)
            anchor = (kl * correct).sum() / correct.sum().clamp_min(1)
            gate_fraction = correct.mean().item()
        self.trace.append({"step": self.step, "norm": norm, "failure": failure,
                           "ema": self.errors.copy(), "ce": ce.detach().item(),
                           "anchor": anchor.detach().item(), "teacher_correct": gate_fraction})
        self.step += 1
        return ce + self.distill_weight * anchor


def my_loss(model, x, y, distill_weight=0.5):
    if not hasattr(model, "_norm_controller"):
        model._norm_controller = AdaptiveNormDistillation(
            copy.deepcopy(model), distill_weight=distill_weight)
    return model._norm_controller(model, x, y)

## Part (a): Single-norm adversarial training

In [ ]:
model_base = load_base_model()
results_base = test_model(model_base)

Starting: base on mps
clean 78.7  linf 48.3  l2 60.0  l1 23.2  union 23.0   (46 s)


In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
results_linf = test_model(model_linf)

Starting: linf_seed0_epochs10 on mps
epoch 1/10: loss 1.2003, 18.7 s
epoch 2/10: loss 1.1738, 16.4 s
epoch 3/10: loss 1.1509, 16.6 s
epoch 4/10: loss 1.1245, 16.6 s
epoch 5/10: loss 1.0965, 16.5 s
epoch 6/10: loss 1.0717, 16.3 s
epoch 7/10: loss 1.0450, 16.2 s
epoch 8/10: loss 1.0312, 16.3 s
epoch 9/10: loss 1.0145, 16.3 s
epoch 10/10: loss 1.0082, 16.3 s
clean 78.7  linf 44.6  l2 57.2  l1 20.2  union 20.1   (44 s)


In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
results_l2 = test_model(model_l2)

Starting: l2_seed0_epochs10 on mps
epoch 1/10: loss 0.8948, 16.4 s
epoch 2/10: loss 0.8330, 16.4 s
epoch 3/10: loss 0.7909, 16.3 s
epoch 4/10: loss 0.7479, 16.3 s
epoch 5/10: loss 0.7065, 16.3 s
epoch 6/10: loss 0.6674, 16.3 s
epoch 7/10: loss 0.6322, 16.3 s
epoch 8/10: loss 0.6111, 16.3 s
epoch 9/10: loss 0.5859, 16.3 s
epoch 10/10: loss 0.5814, 16.4 s
clean 82.1  linf 32.5  l2 59.9  l1 27.6  union 24.3   (44 s)


In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
results_l1 = test_model(model_l1)

Starting: l1_seed0_epochs10 on mps
epoch 1/10: loss 1.2246, 17.4 s
epoch 2/10: loss 0.8840, 17.3 s
epoch 3/10: loss 0.6064, 17.3 s
epoch 4/10: loss 0.4408, 17.3 s
epoch 5/10: loss 0.3323, 17.3 s
epoch 6/10: loss 0.2729, 17.3 s
epoch 7/10: loss 0.2214, 17.3 s
epoch 8/10: loss 0.1857, 17.3 s
epoch 9/10: loss 0.1648, 17.3 s
epoch 10/10: loss 0.1563, 17.2 s
clean 83.8  linf 0.0  l2 0.0  l1 25.8  union 0.0   (44 s)


## Part (b): RAMP

In [ ]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
results_ramp = test_model(model_ramp)

Starting: ramp_seed0_epochs10 on mps
epoch 1/10: loss 1.4267, 36.9 s
epoch 2/10: loss 1.3338, 36.5 s
epoch 3/10: loss 1.3048, 36.5 s
epoch 4/10: loss 1.2759, 36.5 s
epoch 5/10: loss 1.2424, 36.5 s
epoch 6/10: loss 1.2260, 36.5 s
epoch 7/10: loss 1.2002, 36.9 s
epoch 8/10: loss 1.1892, 37.0 s
epoch 9/10: loss 1.1683, 36.9 s
epoch 10/10: loss 1.1693, 37.8 s
clean 78.5  linf 42.7  l2 61.5  l1 45.3  union 40.1   (45 s)


In [ ]:
model_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
results_ramp_1 = test_model(model_ramp_1)

Starting: ramp_seed1_epochs10 on mps
epoch 1/10: loss 1.4325, 37.3 s
epoch 2/10: loss 1.3360, 37.2 s
epoch 3/10: loss 1.2951, 37.5 s
epoch 4/10: loss 1.2735, 37.3 s
epoch 5/10: loss 1.2529, 37.5 s
epoch 6/10: loss 1.2262, 37.3 s
epoch 7/10: loss 1.2069, 37.2 s
epoch 8/10: loss 1.1808, 37.0 s
epoch 9/10: loss 1.1748, 36.7 s
epoch 10/10: loss 1.1689, 36.7 s
clean 78.8  linf 41.6  l2 62.3  l1 45.6  union 40.1   (45 s)


## Part (c): Adaptive norm training

Cycle through Linf, L2, L1, then the norm with the highest smoothed training failure rate.
Add distillation from the frozen base model where its prediction is correct.
The ablation sets the distillation weight to zero.
[Distillation reference: Hinton et al. (2015)](https://arxiv.org/abs/1503.02531)

In [ ]:
model_mine = adv_train(load_base_model(), my_loss)
results_mine = test_model(model_mine)

Starting: proposed_seed0_epochs10 on mps
epoch 1/10: loss 1.2472, 18.4 s
epoch 2/10: loss 1.1812, 18.1 s
epoch 3/10: loss 1.1426, 18.1 s
epoch 4/10: loss 1.1324, 18.4 s
epoch 5/10: loss 1.0869, 19.0 s
epoch 6/10: loss 1.0699, 18.3 s
epoch 7/10: loss 1.0376, 18.6 s
epoch 8/10: loss 1.0270, 18.9 s
epoch 9/10: loss 1.0121, 18.7 s
epoch 10/10: loss 1.0096, 18.7 s
clean 79.4  linf 42.8  l2 61.8  l1 44.9  union 39.5   (45 s)


In [ ]:
model_mine_1 = adv_train(load_base_model(), my_loss, seed=1)
results_mine_1 = test_model(model_mine_1)

Starting: proposed_seed1_epochs10 on mps
epoch 1/10: loss 1.2486, 19.1 s
epoch 2/10: loss 1.1842, 18.7 s
epoch 3/10: loss 1.1487, 18.3 s
epoch 4/10: loss 1.1192, 18.9 s
epoch 5/10: loss 1.0896, 18.5 s
epoch 6/10: loss 1.0616, 19.0 s
epoch 7/10: loss 1.0372, 18.5 s
epoch 8/10: loss 1.0242, 18.6 s
epoch 9/10: loss 1.0114, 18.9 s
epoch 10/10: loss 1.0104, 18.5 s
clean 79.5  linf 42.9  l2 62.7  l1 45.9  union 39.9   (46 s)


In [ ]:
model_ablation = adv_train(load_base_model(), my_loss, distill_weight=0.0)
results_ablation = test_model(model_ablation)

Starting: no_distill_seed0_epochs10 on mps
epoch 1/10: loss 1.2254, 17.9 s
epoch 2/10: loss 1.1581, 17.7 s
epoch 3/10: loss 1.1251, 17.6 s
epoch 4/10: loss 1.1019, 17.5 s
epoch 5/10: loss 1.0696, 17.4 s
epoch 6/10: loss 1.0327, 17.5 s
epoch 7/10: loss 1.0138, 17.5 s
epoch 8/10: loss 0.9936, 17.4 s
epoch 9/10: loss 0.9746, 17.3 s
epoch 10/10: loss 0.9728, 17.4 s
clean 78.4  linf 42.4  l2 62.7  l1 45.6  union 39.4   (44 s)


In [ ]:
model_ablation_1 = adv_train(load_base_model(), my_loss, distill_weight=0.0, seed=1)
results_ablation_1 = test_model(model_ablation_1)

Starting: no_distill_seed1_epochs10 on mps
epoch 1/10: loss 1.2276, 17.8 s
epoch 2/10: loss 1.1602, 17.5 s
epoch 3/10: loss 1.1219, 17.4 s
epoch 4/10: loss 1.0915, 17.3 s
epoch 5/10: loss 1.0635, 17.4 s
epoch 6/10: loss 1.0424, 17.3 s
epoch 7/10: loss 1.0106, 17.4 s
epoch 8/10: loss 0.9965, 17.4 s
epoch 9/10: loss 0.9808, 17.3 s
epoch 10/10: loss 0.9691, 17.4 s
clean 78.6  linf 42.4  l2 62.7  l1 45.3  union 39.2   (44 s)


## Results

In [ ]:
runs = {
    "Base": [(model_base, results_base)],
    "Linf AT": [(model_linf, results_linf)],
    "L2 AT": [(model_l2, results_l2)],
    "L1 AT": [(model_l1, results_l1)],
    "RAMP": [(model_ramp, results_ramp), (model_ramp_1, results_ramp_1)],
    "Proposed": [(model_mine, results_mine), (model_mine_1, results_mine_1)],
    "No distillation": [(model_ablation, results_ablation), (model_ablation_1, results_ablation_1)],
}
rows = []
for name, entries in runs.items():
    row = {"model": name, **{key: np.mean([r[key] for _, r in entries]) for key in results_base}}
    times = [t for model, _ in entries for t in getattr(model, "epoch_times", [])]
    row["s/epoch"] = np.mean(times) if times else np.nan
    rows.append(row)
print(f"{'Model':20s} {'Clean':>7s} {'Linf':>7s} {'L2':>7s} {'L1':>7s} {'Union':>7s} {'s/epoch':>9s}")
for row in rows:
    values = " ".join(f"{row[k]:7.2f}" for k in results_base)
    print(f"{row['model']:20s} {values} {row['s/epoch']:9.2f}")
print("Weakest norms:", {row['model']: [p for p in NORMS if row[p] == min(row[n] for n in NORMS)]
                         for row in rows})


Model                              Clean    Linf      L2      L1   Union   s/epoch
Base                               78.70   48.30   60.00   23.20   23.00        --
Linf AT                            78.70   44.60   57.20   20.20   20.10      16.6
L2 AT                              82.10   32.50   59.90   27.60   24.30      16.3
L1 AT                              83.80    0.00    0.00   25.80    0.00      17.3
RAMP (2 seeds)                     78.65   42.15   61.90   45.45   40.10      37.0
Proposed (2 seeds)                 79.45   42.85   62.25   45.40   39.70      18.6
No distillation (2 seeds)          78.50   42.40   62.70   45.45   39.30      17.5
Weakest norms: {'Base': ['l1'], 'Linf AT': ['l1'], 'L2 AT': ['l1'], 'L1 AT': ['linf', 'l2'], 'RAMP (2 seeds)': ['linf'], 'Proposed (2 seeds)': ['linf'], 'No distillation (2 seeds)': ['linf']}


In [ ]:
# Additional RAMP runs are needed only if the proposed method costs more.
proposed_seconds = [sum(model.epoch_times) for model in (model_mine, model_mine_1)]
ramp_seconds = [sum(model.epoch_times) for model in (model_ramp, model_ramp_1)]
matched_results = []
if np.mean(proposed_seconds) > np.mean(ramp_seconds):
    for seed in (0, 1):
        epochs = max(10, int(np.ceil(10 * proposed_seconds[seed] / ramp_seconds[seed])))
        while True:
            model_matched = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=seed, epochs=epochs)
            elapsed = sum(model_matched.epoch_times)
            if elapsed >= proposed_seconds[seed]:
                break
            epochs = max(epochs + 1, int(np.ceil(epochs * proposed_seconds[seed] / elapsed)))
        result = test_model(model_matched)
        matched_results.append({"seed": seed, "epochs": epochs, "train_seconds": elapsed,
                                "s/epoch": np.mean(model_matched.epoch_times), **result})
    print("Matched-time RAMP:", matched_results)
    print("Mean matched-time RAMP accuracy:",
          {key: np.mean([r[key] for r in matched_results]) for key in results_base})
else:
    print("The proposed method uses less training time than RAMP. No extra runs are needed.")

In [ ]:
best_model, default_results = max(
    [(model_mine, results_mine), (model_mine_1, results_mine_1)],
    key=lambda item: item[1]["union"],
)
strong_results = test_model(best_model, steps=200, restarts=5)
print("Default attacks:", default_results)
print("Stronger attacks:", strong_results)

clean 79.5  linf 42.9  l2 62.5  l1 44.4  union 39.3   (432 s)
Best proposed seed: 1
Default attacks: {'clean': 79.5, 'linf': 42.9, 'l2': 62.7, 'l1': 45.9, 'union': 39.9}
Stronger attacks: {'clean': 79.5, 'linf': 42.9, 'l2': 62.5, 'l1': 44.4, 'union': 39.3}
Accuracy change (stronger - default): {'clean': 0.0, 'linf': 0.0, 'l2': -0.2, 'l1': -1.5, 'union': -0.6}
